# SupplyMind Environment Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShAuRyA-Noodle/Sleep-Token/blob/main/notebooks/01_environment_quickstart.ipynb)

This notebook demonstrates how to interact with the SupplyMind environment — a Gymnasium-compatible RL environment for supply chain risk management.

**Time:** <10 minutes on CPU

In [1]:
# Bootstrap so `import rl` / `from server...` resolve on Colab (git clone) and local Jupyter (sys.path).
import os, sys, subprocess
from pathlib import Path
REPO_URL = 'https://github.com/ShAuRyA-Noodle/Sleep-Token.git'
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    ROOT = Path('/content/Sleep-Token')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)
else:
    # Local Jupyter: notebook lives in notebooks/, so the repo root is the parent dir.
    ROOT = Path.cwd()
    if not (ROOT / 'rl').exists() and (ROOT.parent / 'rl').exists():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Repo root on sys.path: {ROOT}  (Colab={IN_COLAB})')


Repo root on sys.path: c:\Users\Dell\Desktop\Sleep-Token  (Colab=False)


In [2]:
# [nb-exec-patched]
# Colab installs deps here; a local venv already has them. Guarded so a local run
# never downgrades a shared environment (e.g. pinned trl) or requires network.
try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False
if _IN_COLAB:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'gymnasium==0.29.1', 'networkx', 'numpy', 'pydantic', 'fastapi'], check=False)
    print('install ok (Colab)')
else:
    print('Local venv detected — skipping pip install (deps already present).')


Local venv detected — skipping pip install (deps already present).


In [3]:
import gymnasium as gym
import numpy as np
import rl  # Register environments

# Create the easy task environment
env = gym.make('SupplyMind-Easy-v1')
print(f'Observation space: {env.observation_space}')
print(f'Action space: {env.action_space}')

Observation space: Box(-1.0, 2.0, (408,), float32)
Action space: MultiDiscrete([ 7 40])


In [4]:
# Reset and inspect initial state
obs, info = env.reset(seed=42)
print(f'Observation shape: {obs.shape}')
print(f'Action mask shape: {info["action_masks"].shape}')
print(f'Valid actions: {info["action_masks"].sum()} / 280')
print(f'\nFirst 10 obs values: {obs[:10]}')

Observation shape: (408,)
Action mask shape: (280,)
Valid actions: 23 / 280

First 10 obs values: [1. 0. 0. 1. 1. 0. 0. 0. 0. 0.]


In [5]:
# Run a random episode
obs, info = env.reset(seed=42)
total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    if terminated or truncated:
        break

print(f'Episode finished in {steps} steps')
print(f'Total reward: {total_reward:.4f}')

Episode finished in 20 steps
Total reward: 0.4104


In [6]:
# Run the scripted agent
from server.supply_environment import SupplyMindEnvironment
from scripted_agent import choose_action, run_task

env_core = SupplyMindEnvironment()
result = run_task(env_core, 'easy_typhoon_response', seed=42)
print(f'Score: {result["score"]:.4f}')
print(f'Steps: {result["steps_taken"]}')
for k, v in result.get('breakdown', {}).items():
    print(f'  {k}: {v["score"]:.4f} (weight={v["weight"]})')

Score: 0.6722
Steps: 20
  revenue_preserved: 0.6905 (weight=0.3)
  timeliness: 0.6000 (weight=0.25)
  cost_efficiency: 0.1000 (weight=0.15)
  stockout_prevention: 1.0000 (weight=0.1)
  action_coverage: 1.0000 (weight=0.2)


In [7]:
# Compare all 3 tasks
tasks = ['easy_typhoon_response', 'medium_multi_front', 'hard_cascading_crisis']
for task_id in tasks:
    result = run_task(env_core, task_id)
    print(f'{task_id}: score={result["score"]:.4f}, steps={result["steps_taken"]}')

easy_typhoon_response: score=0.7711, steps=20


medium_multi_front: score=0.6962, steps=44


hard_cascading_crisis: score=0.6715, steps=55
